# Sprint 8 — DI, IQueryable Projection, Composition & v1.0 RC

Spec: [`sprint-8-tasks.md`](../../litemapper/docs/requirements/sprint-8-tasks.md) — 13 tasks (S8-T00..T12) shipping the `SmartMapp.Net.DependencyInjection` package and the last v1.0 surface bumps.

| Scope                                        | Task    |
| -------------------------------------------- | ------- |
| `AddSculptor()` core registration            | S8-T01  |
| `AddSculptor(options / lifetime)` overloads  | S8-T02  |
| `IMapper<TOrigin, TTarget>` per-pair         | S8-T03  |
| DI-resolved `IValueProvider<,,>`             | S8-T04  |
| Env-aware `ValidateOnStartup`                | S8-T05  |
| `IQueryable.SelectAs<TTarget>()`             | S8-T06  |
| `MapTo<T>()` / ambient `SelectAs<T>()`       | S8-T07  |
| `Compose<TTarget>(…)` multi-origin           | S8-T08  |
| Samples, benchmarks, NuGet packaging         | S8-T09..T12 |


## Setup


In [ ]:
#r "nuget: Microsoft.EntityFrameworkCore.InMemory, 9.0.0"
#r "nuget: Microsoft.Extensions.Hosting, 9.0.0"
#r "../src/SmartMapp.Net/bin/Release/net10.0/SmartMapp.Net.dll"
#r "../src/SmartMapp.Net.DependencyInjection/bin/Release/net10.0/SmartMapp.Net.DependencyInjection.dll"

using Microsoft.EntityFrameworkCore;
using Microsoft.Extensions.DependencyInjection;
using Microsoft.Extensions.Hosting;
using Microsoft.Extensions.Hosting.Internal;
using Microsoft.Extensions.Logging;
using SmartMapp.Net;
using SmartMapp.Net.Abstractions;
using SmartMapp.Net.DependencyInjection.Extensions;
using SmartMapp.Net.Extensions;

Console.WriteLine("DI + SmartMapp.Net assemblies loaded.");


## 1. `AddSculptor()` — register `ISculptor` + every `IMapper<S, D>` pair


In [ ]:
public sealed class Order    { public int Id { get; init; } public string Buyer { get; init; } = ""; public decimal Amount { get; init; } }
public sealed class OrderDto { public int Id { get; set; } public string Buyer { get; set; } = ""; public decimal Amount { get; set; } public decimal Tax { get; set; } }

var services = new ServiceCollection();
services.AddLogging();
services.AddSculptor(options => options.Bind<Order, OrderDto>(rule => rule
    .Property(d => d.Tax, p => p.From(o => o.Amount * 0.10m))));

using var provider = services.BuildServiceProvider();

var sculptor = provider.GetRequiredService<ISculptor>();
var mapper   = provider.GetRequiredService<IMapper<Order, OrderDto>>();

var src = new Order { Id = 1, Buyer = "Ada", Amount = 100m };
Console.WriteLine($"via ISculptor : Tax={sculptor.Map<Order, OrderDto>(src).Tax:0.00}");
Console.WriteLine($"via IMapper<> : Tax={mapper.Map(src).Tax:0.00}");
Console.WriteLine($"ISculptor is singleton: {object.ReferenceEquals(sculptor, provider.GetRequiredService<ISculptor>())}");


## 2. DI-resolved `IValueProvider<Origin, Target, TMember>` (S8-T04)

`.From<TProvider>()` tells SmartMapp.Net to activate the provider through the `IServiceProvider` on every mapping call — constructor-injected dependencies (loggers, scoped repositories) are honoured.


In [ ]:
public sealed class TaxCalculator : IValueProvider<Order, OrderDto, decimal>
{
    private readonly ILogger<TaxCalculator> _logger;
    public TaxCalculator(ILogger<TaxCalculator> logger) => _logger = logger;
    public decimal Provide(Order o, OrderDto t, string memberName, MappingScope scope)
    {
        var tax = decimal.Round(o.Amount * 0.15m, 2);
        _logger.LogInformation("Tax for order {OrderId}: {Tax}", o.Id, tax);
        return tax;
    }
    object? IValueProvider.Provide(object o, object t, string n, MappingScope s) => Provide((Order)o, (OrderDto)t, n, s);
}

var services = new ServiceCollection();
services.AddLogging(b => b.AddSimpleConsole().SetMinimumLevel(LogLevel.Information));
services.AddSculptor(options => options.Bind<Order, OrderDto>(rule => rule
    .Property(d => d.Tax, p => p.From<TaxCalculator>())));

using var provider = services.BuildServiceProvider();
var sculptor = provider.GetRequiredService<ISculptor>();

var dto = sculptor.Map<Order, OrderDto>(new Order { Id = 42, Buyer = "Grace", Amount = 200m });
Console.WriteLine($"Tax = {dto.Tax:0.00}  (logged above by constructor-injected ILogger)");


## 3. Environment-Aware `ValidateOnStartup` (S8-T05)

`ValidateOnStartup` defaults to `true` in Development and `false` otherwise. The sculptor startup validator short-circuits gracefully in Production unless the user flips the flag explicitly.


In [ ]:
IHostEnvironment Env(string name) => new HostingEnvironment { EnvironmentName = name };

static string Verdict(IHostEnvironment env, bool? explicitSetting)
{
    if (explicitSetting is not null) return explicitSetting.Value ? "validate (explicit)" : "skip (explicit)";
    return env.IsDevelopment() ? "validate (Development default)" : "skip (non-Dev default)";
}

foreach (var (env, opt) in new (IHostEnvironment, bool?)[]
{
    (Env("Development"), null),
    (Env("Staging"),     null),
    (Env("Production"),  null),
    (Env("Production"),  true),
    (Env("Development"), false),
})
    Console.WriteLine($"  {env.EnvironmentName,-12} + explicit={(opt?.ToString() ?? "null"),-5} → {Verdict(env, opt)}");


## 4. `IQueryable.SelectAs<TTarget>(sculptor)` — server-side EF Core projection (S8-T06)

The projection is a single LINQ `Select(…)` expression EF Core can translate to SQL — one `SELECT` with any required `JOIN`s, no N+1.


In [ ]:
public sealed class Ef_Customer { public int Id { get; set; } public string FirstName { get; set; } = ""; public string City { get; set; } = ""; }
public sealed class Ef_Order    { public int Id { get; set; } public int CustomerId { get; set; } public Ef_Customer Customer { get; set; } = default!; public decimal Total { get; set; } }
public sealed class Ef_OrderList
{
    public int     Id                { get; set; }
    public decimal Total             { get; set; }
    public string  CustomerFirstName { get; set; } = "";
    public string  CustomerCity      { get; set; } = "";
}

public sealed class Ef_Db : DbContext
{
    public DbSet<Ef_Customer> Customers => Set<Ef_Customer>();
    public DbSet<Ef_Order>    Orders    => Set<Ef_Order>();
    public Ef_Db(DbContextOptions<Ef_Db> opt) : base(opt) { }
}

var opts = new DbContextOptionsBuilder<Ef_Db>().UseInMemoryDatabase("NB_SP8_" + Guid.NewGuid()).Options;
using var db = new Ef_Db(opts);
var grace = new Ef_Customer { Id = 1, FirstName = "Grace", City = "Arlington" };
var ada   = new Ef_Customer { Id = 2, FirstName = "Ada",   City = "London" };
db.Customers.AddRange(grace, ada);
db.Orders.AddRange(
    new Ef_Order { Id = 10, CustomerId = 1, Total = 100m },
    new Ef_Order { Id = 11, CustomerId = 2, Total = 250m });
db.SaveChanges();

var sculptor = new SculptorBuilder().Configure(o => o.Bind<Ef_Order, Ef_OrderList>(_ => { })).Forge();

var rows = db.Orders
    .Include(o => o.Customer)
    .SelectAs<Ef_Order, Ef_OrderList>(sculptor)
    .OrderBy(r => r.Id)
    .ToList();

foreach (var r in rows)
    Console.WriteLine($"  #{r.Id,-3} total={r.Total,7:0.00}  buyer={r.CustomerFirstName,-8} ({r.CustomerCity})");


## 5. `obj.MapTo<T>()` — ambient, zero-sculptor-argument mapping (S8-T07)

`AddSculptor` installs an `AsyncLocal<ISculptor>` ambient accessor the first time the sculptor resolves. The `MapTo<T>()` extension uses that ambient — clean at the call site.


In [ ]:
var services = new ServiceCollection();
services.AddLogging();
services.AddSculptor(options => options.Bind<Order, OrderDto>(rule => rule
    .Property(d => d.Tax, p => p.From(o => o.Amount * 0.10m))));

using var provider = services.BuildServiceProvider();
_ = provider.GetRequiredService<ISculptor>();  // triggers the ambient install

var order = new Order { Id = 5, Buyer = "Alan", Amount = 500m };
var dto   = order.MapTo<OrderDto>();            // no sculptor argument
Console.WriteLine($"Ambient MapTo<OrderDto>() → Id={dto.Id}, Buyer={dto.Buyer}, Tax={dto.Tax:0.00}");


## 6. `Compose<TTarget>(origin1, origin2, …)` — multi-origin composition (S8-T08)

Register a composition rule declaring the origin slots, then dispatch at runtime with any caller order. Null origins are skipped, single-origin `Compose<T>(x)` is identical to `Map<S, T>(x)`.


In [ ]:
public sealed class User         { public int    UserId      { get; init; } public string DisplayName { get; init; } = ""; }
public sealed class OrderSummary { public int    OpenOrders  { get; init; } public decimal LifetimeValue { get; init; } }
public sealed class Dashboard
{
    public int     UserId        { get; set; }
    public string  DisplayName   { get; set; } = "";
    public int     OpenOrders    { get; set; }
    public decimal LifetimeValue { get; set; }
}

var sculptor = new SculptorBuilder()
    .Configure(o => o.Compose<Dashboard>(c => c
        .FromOrigin<User>()
        .FromOrigin<OrderSummary>()))
    .Forge();

var usr = new User         { UserId = 1, DisplayName = "Ada" };
var sum = new OrderSummary { OpenOrders = 3, LifetimeValue = 1500m };

var forward  = sculptor.Compose<Dashboard>(usr, sum);
var reversed = sculptor.Compose<Dashboard>(sum, usr);
var partial  = sculptor.Compose<Dashboard>(usr, null!);

Console.WriteLine($"forward : UserId={forward.UserId},  DisplayName={forward.DisplayName},  OpenOrders={forward.OpenOrders},  LifetimeValue={forward.LifetimeValue}");
Console.WriteLine($"reversed: UserId={reversed.UserId}, DisplayName={reversed.DisplayName}, OpenOrders={reversed.OpenOrders}, LifetimeValue={reversed.LifetimeValue}");
Console.WriteLine($"partial : UserId={partial.UserId},  DisplayName={partial.DisplayName},  OpenOrders={partial.OpenOrders}   (null Summary → slot skipped)");


## Next

- **`99-acceptance-tests.ipynb`** — assertion-driven notebook that throws if any feature in notebooks 01–08 regresses. Run it after any refactor as a one-button smoke test.
